# Práctica 5

## Autora
[Andrea Santana López]("https://github.com/AndreaSantaloz")

## Arte Ticktoker

In [1]:
import cv2
import dlib
import numpy as np
import imutils
import os



def mask_feature(image, pts, x_crop, y_crop, w_crop, h_crop, blur_kernel=11):
    feature_mask = np.zeros((h_crop, w_crop), dtype=np.uint8)

    pts_rel = pts.copy()
    pts_rel[:, 0] -= x_crop
    pts_rel[:, 1] -= y_crop

    cv2.fillConvexPoly(feature_mask, pts_rel, 255)
    feature_mask_soft = cv2.GaussianBlur(feature_mask, (blur_kernel, blur_kernel), 0)

    crop = image[y_crop:y_crop + h_crop, x_crop:x_crop + w_crop]

    return crop, feature_mask_soft  


try:
    BASE = os.path.dirname(os.path.abspath(__file__))
except:
    BASE = os.getcwd()

SHAPE_PREDICTOR_PATH = os.path.join(
    BASE,
    "shape_predictor_68_face_landmarks.dat",
    "shape_predictor_68_face_landmarks.dat"
)

if not os.path.exists(SHAPE_PREDICTOR_PATH):
    print(" No se encuentra el predictor en:", SHAPE_PREDICTOR_PATH)
    exit()



SCALE = 2.0       
HCOMP = 0.4       
FACE_Y = 0.001     
CROP = 15
BLUR_FACE = (35, 35)
BLUR_FEATURE = 9
EYE_GAP_FACTOR = 0  



det = dlib.get_frontal_face_detector()
pred = dlib.shape_predictor(SHAPE_PREDICTOR_PATH)

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print(" No se pudo abrir la cámara.")
    exit()

print("Presiona 'q' para salir")



while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame = imutils.resize(frame, width=500)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    output = frame.copy()

    rects = det(gray, 0)
    

    for rect in rects:
        pts = np.array([[pred(gray, rect).part(i).x, pred(gray, rect).part(i).y] for i in range(68)])

        x, y = rect.left(), rect.top()
        w = rect.right() - x
        h = rect.bottom() - y

        face_mask = np.zeros(frame.shape[:2], dtype="uint8")
        cv2.ellipse(face_mask, (x + w//2, y + h//2), (w//2, int(h*0.55)), 0, 0, 360, 255, -1)

        blurred = cv2.GaussianBlur(frame, BLUR_FACE, 0)

        mask3 = cv2.cvtColor(face_mask, cv2.COLOR_GRAY2BGR)
        inv = cv2.bitwise_not(mask3)

        output = cv2.add(
            cv2.bitwise_and(blurred, mask3),
            cv2.bitwise_and(frame, inv)
        )


        features = {
            "left_eye": pts[36:42],
            "right_eye": pts[42:48],
            "mouth": pts[48:68]
        }

        cx = x + w // 2
        base_y = int(y + h * FACE_Y)
        eye_gap = int(w * EYE_GAP_FACTOR)

        for name, fpts in features.items():

            x_min, y_min = np.min(fpts, axis=0)
            x_max, y_max = np.max(fpts, axis=0)

            xc = max(0, x_min - CROP)
            yc = max(0, y_min - CROP)
            wc = (x_max - x_min) + CROP * 2
            hc = (y_max - y_min) + CROP * 2

            feat, mask_soft = mask_feature(frame, fpts, xc, yc, wc, hc, BLUR_FEATURE)
            if feat.size == 0:
                continue

            # ESCALADO
            sw = SCALE * (HCOMP if name != "mouth" else 1)
            sh = SCALE

            nw = int(feat.shape[1] * sw)
            nh = int(feat.shape[0] * sh)
            
            feat_res = cv2.resize(feat, (nw, nh))
            mask_res = cv2.resize(mask_soft, (nw, nh))
            mask_res = mask_res.astype(float) / 255.0
            mask_res = np.dstack([mask_res] * 3)

            # NUEVA POSICIÓN
            if name == "left_eye":
                nx = cx - nw - eye_gap
                ny = base_y - nh // 2
            elif name == "right_eye":
                nx = cx + eye_gap
                ny = base_y - nh // 2
            elif name == "mouth":
                nx = cx - nw // 2
                ny = base_y + int(h * 0.05) 

            # BLEND
            y1 = max(0, ny)
            y2 = min(output.shape[0], ny + nh)
            x1 = max(0, nx)
            x2 = min(output.shape[1], nx + nw)

            region = output[y1:y2, x1:x2]
            hp, wp = region.shape[:2]

            feat_res = feat_res[:hp, :wp]
            mask_res = mask_res[:hp, :wp]

            blended = (region * (1 - mask_res) + feat_res * mask_res).astype(np.uint8)
            output[y1:y2, x1:x2] = blended

    cv2.imshow("Filtro Compactado", output)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


Presiona 'q' para salir
